# EDA: корпус «Русская Истина» (теги: истина, правда, ложь)

Разведочный анализ корпуса статей сайта «Русская Истина», отобранных по тегам **истина / правда / ложь**.

- **Источник данных:** `Сайт_Русская_Истина_Статьи_по_тегам_истина_правда_ложь.xlsx`
- **Отчёт:** [`docs/eda_russkaya_istina_report.md`](docs/eda_russkaya_istina_report.md)
- **Навигатор по проекту:** [`docs/project_guide.md`](docs/project_guide.md)
- **Автогенерация артефактов:** `python scripts/run_eda_russkaya_istina.py` → `output/eda_russkaya_istina/`

In [ ]:
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
from IPython.display import Image, display, Markdown

%matplotlib inline
plt.rcParams["figure.figsize"] = (10, 5)

ROOT = Path(".").resolve()
XLSX = ROOT / "Сайт_Русская_Истина_Статьи_по_тегам_истина_правда_ложь.xlsx"
OUT = ROOT / "output/eda_russkaya_istina"
FIG = OUT / "figures"
PATHS = {
    "constructor": (
        ROOT / "data/graphs/russkaya_istina/constructor/discourse/nodes.csv",
        ROOT / "data/graphs/russkaya_istina/constructor/discourse/edges.csv",
    ),
    "graphrag_baseline": (
        ROOT / "data/graphs/russkaya_istina/constructor/graphrag_baseline/nodes.csv",
        ROOT / "data/graphs/russkaya_istina/constructor/graphrag_baseline/edges.csv",
    ),
    "invariant_core": (
        ROOT / "data/graphs/russkaya_istina/constructor/invariant_core/nodes.csv",
        ROOT / "data/graphs/russkaya_istina/constructor/invariant_core/edges.csv",
    ),
}

## 0. Обновить артефакты EDA (опционально)

Пересоздаёт фигуры, JSON и проверки адекватности в `output/eda_russkaya_istina/`.

In [ ]:
RUN_EDA = True  # поставьте False, если артефакты уже свежие

if RUN_EDA:
    r = subprocess.run(
        [sys.executable, str(ROOT / "scripts/run_eda_russkaya_istina.py")],
        cwd=ROOT, capture_output=True, text=True
    )
    print(r.stdout)
    if r.returncode != 0:
        print(r.stderr)

## 1. Проверки адекватности (sanity checks)

In [ ]:
sanity = json.loads((OUT / "sanity_checks.json").read_text(encoding="utf-8"))
corpus_stats = json.loads((OUT / "corpus_stats.json").read_text(encoding="utf-8"))
graph_metrics = json.loads((OUT / "graph_metrics.json").read_text(encoding="utf-8"))

df_checks = pd.DataFrame(sanity["checks"])
display(df_checks)
print(f"\nИтого: {sanity['passed']}/{sanity['total']} проверок пройдено")
assert sanity["passed"] >= sanity["total"] - 1, "Слишком много провалов — проверьте корпус"

## 2. Корпус: сводка

In [ ]:
display(Markdown(f"""
| Метрика | Значение |
|---------|----------|
| Статей | {corpus_stats['n_articles']} |
| Авторов | {corpus_stats['n_authors']} |
| Медиана длины аннотации | {corpus_stats['median_summary_len']:.0f} символов |
| Период | {corpus_stats['year_min']}–{corpus_stats['year_max']} |
"""))

print("Топ авторов:")
display(pd.DataFrame(corpus_stats["top_authors"].items(), columns=["автор", "статей"]))

In [ ]:
# Загружаем исходные данные для интерактивного анализа
def parse_tags(val):
    if pd.isna(val) or val is True or val is False:
        return []
    s = str(val).strip().strip("'\"")
    return [t.strip() for t in s.split(",") if t.strip()]

df = pd.read_excel(XLSX, sheet_name="Лист1", engine="openpyxl")
df.columns = ["date", "author", "title", "url", "summary", "tags"]
df["tags_list"] = df["tags"].apply(parse_tags)
df["n_tags"] = df["tags_list"].apply(len)
df["summary_len"] = df["summary"].astype(str).str.len()
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["year"] = df["date"].dt.year
df = df[df["title"].notna() & (df["title"].astype(str).str.strip() != "nan")].reset_index(drop=True)

print(f"Загружено статей: {len(df)}")
display(df[["date", "author", "title", "n_tags", "summary_len"]].head(10))

In [ ]:
# Фигуры корпуса
for name in ["01_corpus_distributions.png", "02_authors_tags.png",
             "03_domain_words.png", "04_temporal_dynamics.png"]:
    p = FIG / name
    if p.exists():
        display(Image(filename=str(p)))

## 3. Теги и доменная валидность

In [ ]:
all_tags = [t for tl in df["tags_list"] for t in tl]
tag_c = Counter(all_tags)

print("Все теги (частота):")
display(pd.DataFrame(tag_c.most_common(), columns=["тег", "упоминаний"]))

print("\nДоменные слова в аннотациях:")
domain_words = ["истина", "правда", "ложь", "обман", "постправда",
                "релятивизм", "честность", "заблуждение"]
wc = {w: int(df["summary"].astype(str).str.lower().str.count(w).sum()) for w in domain_words}
display(pd.DataFrame([(k, v) for k, v in wc.items() if v > 0], columns=["слово", "упоминаний"]))

## 4. Временная динамика

In [ ]:
year_counts = df["year"].value_counts().sort_index()
print("Публикации по годам:")
display(pd.DataFrame(year_counts).rename(columns={"year": "статей"}))

print("\nАвторы по годам (топ-5 авторов):")
top5 = df["author"].value_counts().head(5).index.tolist()
display(df[df["author"].isin(top5)].groupby(["year", "author"]).size().unstack(fill_value=0))

## 5. Сравнение графов

In [ ]:
rows = []
for name, m in graph_metrics.items():
    if m:
        rows.append({"граф": name, **m})
display(pd.DataFrame(rows).set_index("граф"))

p = FIG / "05_graph_comparison.png"
if p.exists():
    display(Image(filename=str(p)))

## 6. Визуализация сетей и сообществ

In [ ]:
for gname in ["constructor", "invariant_core", "graphrag_baseline"]:
    net = FIG / f"08_network_{gname}.png"
    comm = FIG / f"09_communities_{gname}.png"
    if net.exists():
        display(Markdown(f"### {gname}"))
        display(Image(filename=str(net)))
        if comm.exists():
            display(Image(filename=str(comm)))

## 7. Конструктор: центральность и «жареные» связи

In [ ]:
npth, epth = PATHS["constructor"]
edges = pd.read_csv(epth)
G = nx.from_pandas_edgelist(edges, "source", "target", edge_attr=True)

deg = sorted(G.degree(weight="weight"), key=lambda x: x[1], reverse=True)[:15]
bet = sorted(nx.betweenness_centrality(G, weight="weight").items(), key=lambda x: x[1], reverse=True)[:15]
print("Топ degree:", deg)
print("\nТоп betweenness:", bet)

if "surprisal" in edges.columns:
    print("\nТоп surprisal (жареные связи):")
    cols = [c for c in ["source", "target", "surprisal", "weight", "methods"] if c in edges.columns]
    display(edges.nlargest(10, "surprisal")[cols])

if "methods" in edges.columns:
    rhet = edges[edges["methods"].astype(str).str.contains("rhetorical", na=False)]
    print(f"\nРиторических рёбер: {len(rhet)}")
    if len(rhet):
        cols_r = [c for c in ["source", "target", "relation", "weight"] if c in rhet.columns]
        display(rhet.head(8)[cols_r])

In [ ]:
for name in ["06_degree_constructor.png", "07_threshold_constructor.png"]:
    p = FIG / name
    if p.exists():
        display(Image(filename=str(p)))

## 8. Инвариантное ядро vs конструктор

In [ ]:
npth_inv, epth_inv = PATHS["invariant_core"]
if epth_inv.exists():
    edges_inv = pd.read_csv(epth_inv)
    # edge_attr только если есть дополнительные колонки помимо source/target
    extra_cols = [c for c in edges_inv.columns if c not in ("source", "target")]
    G_inv = nx.from_pandas_edgelist(
        edges_inv, "source", "target",
        edge_attr=extra_cols if extra_cols else None
    )

    # Узлы, общие с конструктором
    common_nodes = set(G.nodes()) & set(G_inv.nodes())
    print(f"Узлов в конструкторе: {G.number_of_nodes()}")
    print(f"Узлов в инвариантном ядре: {G_inv.number_of_nodes()}")
    print(f"Общих узлов: {len(common_nodes)}")
    print(f"Рёбер в конструкторе: {G.number_of_edges()}")
    print(f"Рёбер в инвариантном ядре: {G_inv.number_of_edges()}")

    # Топ узлов инвариантного ядра (по степени, без весов если их нет)
    weight_kw = {"weight": "weight"} if "weight" in edges_inv.columns else {}
    deg_inv = sorted(G_inv.degree(**weight_kw), key=lambda x: x[1], reverse=True)[:10]
    print("\nТоп degree (инвариантное ядро):", deg_inv)

## 9. Полный список статей

In [ ]:
display(df[["date", "author", "title", "tags"]].sort_values("date").reset_index(drop=True))